# Google Play Review Feature Engineering v0

This notebook creates a lightweight, rule-based feature layer from the validated Phase 2 Google Play review database.

The scope is intentionally limited:

- continue using the existing ingestion monitoring layer during normal collection runs
- generate transparent review-level features from cleaned review data
- create basic app-level aggregates
- document every feature and its calculation
- export a small sample and a reliability assessment
- do **not** train a sentiment, topic, or machine-learning model

Operationally, feature generation should run only after ingestion and hard-failure validation complete. A monitoring warning can be documented and reviewed, but a hard failure should stop downstream feature use until the run is corrected.


## 1. Imports and configuration

The notebook uses only the Python standard library, SQLite, and pandas. No external language model or sentiment package is required. The non-English indicator is therefore a conservative, documented heuristic rather than a full language-identification model.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import re
import shutil
import sqlite3
import textwrap
import zipfile

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_NAME = "Google Play Review Feature Engineering v0"
EXPECTED_APP_COUNT = 10
EXPECTED_REVIEW_ROWS = 34_601
SHORT_REVIEW_CHAR_THRESHOLD = 20
LOW_SIGNAL_MAX_WORDS = 2
LOW_SIGNAL_MIN_ALNUM_CHARS = 10

WORK_DIR = Path(
    os.environ.get(
        "FEATURE_ENGINEERING_WORKDIR",
        "/content/google_play_feature_engineering_v0",
    )
)
SOURCE_DIR = WORK_DIR / "source"
EXTRACT_DIR = WORK_DIR / "source_extracted"
DATABASE_DIR = WORK_DIR / "database"
OUTPUT_DIR = WORK_DIR / "outputs"
REPORT_DIR = WORK_DIR / "reports"
PACKAGE_DIR = WORK_DIR / "github_upload_files"

for directory in [
    WORK_DIR,
    SOURCE_DIR,
    EXTRACT_DIR,
    DATABASE_DIR,
    OUTPUT_DIR,
    REPORT_DIR,
    PACKAGE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Work directory:", WORK_DIR)
print("Configured review-row expectation:", f"{EXPECTED_REVIEW_ROWS:,}")


Work directory: /mnt/data/feature_engineering_v0_run
Configured review-row expectation: 34,601


## 2. Upload or locate the completed Phase 2 source package

Use the complete Run B follow-up package. The notebook validates the package manifest before reading the database.


In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

source_override = os.environ.get("FEATURE_SOURCE_PACKAGE")

if source_override:
    SOURCE_PACKAGE = Path(source_override).expanduser().resolve()
else:
    SOURCE_PACKAGE = None
    try:
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Please upload exactly one Phase 2 source ZIP.")
        uploaded_name = next(iter(uploaded))
        SOURCE_PACKAGE = Path(uploaded_name).resolve()
    except ImportError:
        candidates = sorted(
            list(Path.cwd().glob("phase2_cadence_runB_followup_github_upload_files_complete_report_*.zip"))
            + list(Path("/mnt/data").glob("phase2_cadence_runB_followup_github_upload_files_complete_report_*.zip"))
        )
        if len(candidates) != 1:
            raise FileNotFoundError(
                "Could not identify exactly one complete Run B follow-up ZIP. "
                "Set FEATURE_SOURCE_PACKAGE or place the ZIP in the working directory."
            )
        SOURCE_PACKAGE = candidates[0].resolve()

if not SOURCE_PACKAGE.exists():
    raise FileNotFoundError(f"Source package not found: {SOURCE_PACKAGE}")

source_copy = SOURCE_DIR / SOURCE_PACKAGE.name
if SOURCE_PACKAGE != source_copy:
    shutil.copy2(SOURCE_PACKAGE, source_copy)
SOURCE_PACKAGE = source_copy

print("Source package:", SOURCE_PACKAGE.name)
print("Package size (MB):", round(SOURCE_PACKAGE.stat().st_size / (1024 ** 2), 2))
print("Package SHA-256:", sha256_file(SOURCE_PACKAGE))


Source package: phase2_cadence_runB_followup_github_upload_files_complete_report_20260714_170311_utc(2).zip
Package size (MB): 21.48
Package SHA-256: 0baafebdc1afbecfec32036c4b4f4154a7ec6254cbbf38621ad5eeeb4764a205


## 3. Validate and extract the source package

The package-level manifest is checked for file existence, file size, and SHA-256. The nested SQLite archive is then extracted and checked against the source metadata.


In [3]:
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(SOURCE_PACKAGE, "r") as source_zip:
    source_zip.extractall(EXTRACT_DIR)

manifest_path = EXTRACT_DIR / "final_package_manifest.csv"
metadata_path = EXTRACT_DIR / "phase2_cadence_runB_followup_metadata.json"
database_archive_path = EXTRACT_DIR / "google_play_reviews_after_runB_followup.sqlite.zip"

required_source_files = [
    manifest_path,
    metadata_path,
    database_archive_path,
    EXTRACT_DIR / "phase2_cadence_runB_followup_final_database_validation.csv",
]
missing_required = [path.name for path in required_source_files if not path.exists()]
if missing_required:
    raise FileNotFoundError(f"Required source files are missing: {missing_required}")

source_manifest_df = pd.read_csv(manifest_path)
source_validation_rows = []

for row in source_manifest_df.itertuples(index=False):
    file_path = EXTRACT_DIR / row.file_name
    exists = file_path.exists()
    actual_size = file_path.stat().st_size if exists else 0
    actual_hash = sha256_file(file_path) if exists else ""
    source_validation_rows.append({
        "file_name": row.file_name,
        "exists": exists,
        "expected_size_bytes": int(row.size_bytes),
        "actual_size_bytes": actual_size,
        "size_matches": exists and actual_size == int(row.size_bytes),
        "expected_sha256": str(row.sha256),
        "actual_sha256": actual_hash,
        "hash_matches": exists and actual_hash == str(row.sha256),
    })

source_validation_df = pd.DataFrame(source_validation_rows)
if not source_validation_df[["exists", "size_matches", "hash_matches"]].all().all():
    failed = source_validation_df.loc[
        ~source_validation_df[["exists", "size_matches", "hash_matches"]].all(axis=1),
        ["file_name", "exists", "size_matches", "hash_matches"],
    ]
    raise ValueError(f"Source-manifest validation failed:\n{failed}")

source_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

if DATABASE_DIR.exists():
    shutil.rmtree(DATABASE_DIR)
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(database_archive_path, "r") as database_zip:
    database_zip.extractall(DATABASE_DIR)

database_candidates = list(DATABASE_DIR.glob("*.sqlite"))
if len(database_candidates) != 1:
    raise ValueError("Expected exactly one SQLite database inside the database archive.")
DB_PATH = database_candidates[0]

actual_database_hash = sha256_file(DB_PATH)
expected_database_hash = str(source_metadata["database_sha256"])
if actual_database_hash != expected_database_hash:
    raise ValueError("The extracted SQLite database does not match the metadata SHA-256.")

print("Source package validated successfully.")
print("Manifest files validated:", len(source_validation_df))
print("Database:", DB_PATH.name)
print("Database size (MB):", round(DB_PATH.stat().st_size / (1024 ** 2), 2))
print("Database SHA-256:", actual_database_hash)
display(source_validation_df[["file_name", "exists", "size_matches", "hash_matches"]])


Source package validated successfully.
Manifest files validated: 22
Database: google_play_reviews_after_runB_followup.sqlite
Database size (MB): 84.68
Database SHA-256: a26f6db1b16b35d7763fb8f6e9ce4048ecd3b947749811cc5c34c3d099f3cd69


,file_name,exists,size_matches,hash_matches
0,google_play_reviews_after_runB_followup.sqlite.zip,True,True,True
1,phase2_cadence_runA_runB_app_level_recommendations.csv,True,True,True
2,phase2_cadence_runA_runB_comparison_validation.csv,True,True,True
3,phase2_cadence_runA_runB_final_report.md,True,True,True
4,phase2_cadence_runA_runB_final_report_validation.csv,True,True,True
5,phase2_cadence_runA_runB_run_level_comparison.csv,True,True,True
6,phase2_cadence_runA_timestamp_audit_reconstructed.csv,True,True,True
7,phase2_cadence_runA_timestamp_summary_reconstructed.csv,True,True,True
8,phase2_cadence_runA_timestamp_validation.csv,True,True,True
9,phase2_cadence_runB_followup_app_summary.csv,True,True,True


## 4. Validate the database snapshot

This section checks the core database conditions that must pass before feature generation. These checks are treated as hard-failure guards.


In [4]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")


def scalar(query):
    return conn.execute(query).fetchone()[0]

raw_rows = int(scalar("SELECT COUNT(*) FROM phase2_reviews_raw"))
cleaned_rows = int(scalar("SELECT COUNT(*) FROM phase2_reviews_cleaned"))
app_count = int(scalar("SELECT COUNT(*) FROM phase2_apps"))
completed_runs = int(scalar("SELECT COUNT(*) FROM phase2_ingestion_runs WHERE status='completed'"))
app_summary_rows = int(scalar("SELECT COUNT(*) FROM phase2_app_run_summary"))
quality_flag_rows = int(scalar("SELECT COUNT(*) FROM phase2_quality_flags"))

duplicate_identity_groups = int(scalar("""
    SELECT COUNT(*)
    FROM (
        SELECT source, app_id, review_id
        FROM phase2_reviews_raw
        GROUP BY source, app_id, review_id
        HAVING COUNT(*) > 1
    )
"""))
raw_without_cleaned = int(scalar("""
    SELECT COUNT(*)
    FROM phase2_reviews_raw r
    LEFT JOIN phase2_reviews_cleaned c USING(review_key)
    WHERE c.review_key IS NULL
"""))
cleaned_without_raw = int(scalar("""
    SELECT COUNT(*)
    FROM phase2_reviews_cleaned c
    LEFT JOIN phase2_reviews_raw r USING(review_key)
    WHERE r.review_key IS NULL
"""))
foreign_key_violations = len(conn.execute("PRAGMA foreign_key_check").fetchall())

source_guard_rows = [
    ("expected_raw_review_rows", raw_rows == EXPECTED_REVIEW_ROWS, raw_rows),
    ("expected_cleaned_review_rows", cleaned_rows == EXPECTED_REVIEW_ROWS, cleaned_rows),
    ("raw_and_cleaned_rows_match", raw_rows == cleaned_rows, raw_rows - cleaned_rows),
    ("expected_10_apps", app_count == EXPECTED_APP_COUNT, app_count),
    ("six_completed_phase2_runs", completed_runs == 6, completed_runs),
    ("sixty_app_run_summary_rows", app_summary_rows == 60, app_summary_rows),
    ("no_duplicate_review_identities", duplicate_identity_groups == 0, duplicate_identity_groups),
    ("no_raw_rows_without_cleaned", raw_without_cleaned == 0, raw_without_cleaned),
    ("no_cleaned_rows_without_raw", cleaned_without_raw == 0, cleaned_without_raw),
    ("no_foreign_key_violations", foreign_key_violations == 0, foreign_key_violations),
]
source_guard_df = pd.DataFrame(
    source_guard_rows,
    columns=["validation_check", "passed", "observed_value"],
)

if not source_guard_df["passed"].all():
    raise ValueError(
        "A hard-failure database guard did not pass:\n"
        + source_guard_df.loc[~source_guard_df["passed"]].to_string(index=False)
    )

print("Database snapshot passed all hard-failure guards.")
print("Raw / cleaned rows:", f"{raw_rows:,}")
print("Apps:", app_count)
print("Completed runs:", completed_runs)
print("Quality flags:", f"{quality_flag_rows:,}")
display(source_guard_df)


Database snapshot passed all hard-failure guards.
Raw / cleaned rows: 34,601
Apps: 10
Completed runs: 6
Quality flags: 75,918


,validation_check,passed,observed_value
0,expected_raw_review_rows,True,34601
1,expected_cleaned_review_rows,True,34601
2,raw_and_cleaned_rows_match,True,0
3,expected_10_apps,True,10
4,six_completed_phase2_runs,True,6
5,sixty_app_run_summary_rows,True,60
6,no_duplicate_review_identities,True,0
7,no_raw_rows_without_cleaned,True,0
8,no_cleaned_rows_without_raw,True,0
9,no_foreign_key_violations,True,0


## 5. Load the cleaned review feature base

Only fields needed for feature engineering and traceability are loaded. User names, user images, and raw JSON are excluded from the exported feature table.


In [5]:
review_query = """
SELECT
    r.review_key,
    r.source,
    r.app_id,
    r.app_name,
    r.review_id,
    c.content_cleaned,
    c.score,
    c.has_developer_reply,
    r.review_created_at,
    r.fetched_at,
    c.cleaned_at,
    r.app_version,
    r.run_id
FROM phase2_reviews_raw AS r
INNER JOIN phase2_reviews_cleaned AS c
    ON r.review_key = c.review_key
ORDER BY r.app_name, r.review_created_at, r.review_key
"""

reviews_df = pd.read_sql_query(review_query, conn)
conn.close()

if len(reviews_df) != EXPECTED_REVIEW_ROWS:
    raise ValueError("Loaded review rows do not match the validated database total.")

print("Feature-base rows:", f"{len(reviews_df):,}")
print("Unique review keys:", f"{reviews_df['review_key'].nunique():,}")
print("Apps:", reviews_df["app_name"].nunique())
display(reviews_df.head(5))


Feature-base rows: 34,601
Unique review keys: 34,601
Apps: 10


,review_key,source,app_id,app_name,review_id,content_cleaned,score,has_developer_reply,review_created_at,fetched_at,cleaned_at,app_version,run_id
0,8700e0bccfc44936acc518dab7ffa6ce9aae9dfe5cd14b3cbdb3107ead5a56f6,google_play,com.dd.doordash,DoorDash,0aaf6b5c-9619-4576-8152-fac88d477570,They now charge for a Regular delivery 2.99 or wait on a 30 minute delivery for free cold food at a High price.. uni...,1,0,2026-06-28T23:28:16+00:00,2026-07-08T03:44:59.159630+00:00,2026-07-08T03:44:59.990307+00:00,15.281.1,phase2_day1_controlled_scale_20260708_034445
1,f40e0f6bd214601508a541dfbf7e6039d7857433844f8341ea1ac9d846fec414,google_play,com.dd.doordash,DoorDash,659e7426-23ec-4e60-847a-a337580a9494,Update: This app STILL deserves 1 star Loved this app! But I had a 40% off promo code that disappeared. The whole pr...,1,0,2026-06-28T23:30:27+00:00,2026-07-08T03:44:59.159630+00:00,2026-07-08T03:44:59.990213+00:00,15.281.1,phase2_day1_controlled_scale_20260708_034445
2,0774835d100ffc9bfa3b14e1f6195708721493fcb3010cfdf731dd47d63fd252,google_play,com.dd.doordash,DoorDash,4e435abe-e4a4-4e1e-ab87-0a0cf9d93d24,"Over priced app that always crashes. Use any other app, save your time and money.",1,0,2026-06-28T23:30:42+00:00,2026-07-08T03:44:59.159630+00:00,2026-07-08T03:44:59.990101+00:00,15.187.9,phase2_day1_controlled_scale_20260708_034445
3,02e837ed23ecc2c4cb4eb0e862fb856e7743df6bcdca5b210f6fe29cdd1fd841,google_play,com.dd.doordash,DoorDash,4f96af3b-e555-411e-9353-33b813fe1cfc,I keep getting an error message and I getting pissed off,1,0,2026-06-28T23:34:50+00:00,2026-07-08T03:44:59.159630+00:00,2026-07-08T03:44:59.990035+00:00,15.276.7,phase2_day1_controlled_scale_20260708_034445
4,660e98ecca88ba5086075447c3122900c6c22b3eb57d6befcba973cf9c4da5e4,google_play,com.dd.doordash,DoorDash,0bdc7e18-70d3-440d-8b23-7dc3c803cd64,trash,1,0,2026-06-28T23:34:57+00:00,2026-07-08T03:44:59.159630+00:00,2026-07-08T03:44:59.989973+00:00,15.270.0,phase2_day1_controlled_scale_20260708_034445


## 6. Define transparent feature rules

The first version uses fixed, readable rules:

- rating group: `low` for 1–2, `middle` for 3, and `high` for 4–5
- short review: fewer than 20 cleaned-text characters
- low signal: at most 2 word tokens or fewer than 10 alphanumeric characters
- duplicate identity: repeated `source + app_id + review_id`
- repeated text: the same normalized cleaned text appears more than once within the same app
- non-English: a conservative script/marker heuristic with an explicit `undetermined` state
- issue indicators: exact keyword/phrase groups, not sentiment or topic-model predictions


In [6]:
WORD_RE = re.compile(r"[^\W_]+(?:['’\-][^\W_]+)*", re.UNICODE)

ENGLISH_MARKERS = set("""
the a an and or but if then this that these those it its is are was were be been being
to of in on for with from at by as not no yes do does did can could would should will
just very really too so app apps good great love like use using used work works working
dont doesn't didn can't wont won't have has had my me i you your we our they their them
he she what why how when where because after before more most much many one two new old
please thanks thank get got getting make makes made keep update updated version review
reviews phone video videos music account login payment service support ads ad time day
""".lower().split())

LANGUAGE_MARKERS = {
    "es": set("que los las una para porque muy gracias bueno buena funciona aplicación aplicacion cuenta pago actualizar actualización actualizacion anuncios servicio recomiendo pero esto esta cómo como tiene todo nada puedo deja peor mejor usuario usuarios".split()),
    "fr": set("les une pour avec sans très tres merci bonne fonctionne compte paiement pourquoi cette jamais toujours beaucoup application mise jour mauvais meilleure utilisateur utilisateurs".split()),
    "pt": set("uma para com sem muito não nao obrigado obrigada boa melhor funciona aplicativo conta pagamento atualização atualizacao anúncios anuncios serviço servico recomendo porque esta isso tudo nada posso".split()),
    "de": set("der die das ein eine und oder aber für fur mit ohne dass ist sind sehr nicht mein warum danke gut besser schlecht funktioniert konto zahlung aktualisierung anwendung benutzer".split()),
    "it": set("gli una per con senza che sono molto grazie buona migliore peggiore funziona applicazione pagamento aggiornamento perché perche questa questo utenti servizio".split()),
    "id": set("saya kamu yang dan atau tapi untuk dengan tanpa tidak sangat ini itu karena dimana terima kasih bagus baik lebih buruk aplikasi akun pembayaran pembaruan tolong pengguna".split()),
    "ro": set("foarte este sunt pentru fără fara mulțumesc multumesc bun rău rau funcționează functioneaza aplicație aplicatie cont plată plata actualizare există exista serviciu".split()),
    "tr": set("çok cok değil degil evet hayır hayir için icin ile bir bu şu su teşekkür tesekkur iyi kötü kotu çalışmıyor calismiyor uygulama hesabı hesabi ödeme odeme güncelleme guncelleme".split()),
    "pl": set("jest nie bardzo dla oraz ale aplikacja konto płatność platnosc aktualizacja działa dziala dziękuję dziekuje użytkownik uzytkownik".split()),
    "tl": set("ang mga para pero hindi walang ito ako salamat maganda kailangan ayaw gusto dahil sana lang naman talaga".split()),
    "hi_roman": set("hai nahi nahin mera meri mere mujhe kya kyon kyun bahut bhut acha accha karo kar raha rha mein se ka ki ke ko wali wala ho gaya gya nhi".split()),
}

NON_LATIN_RANGES = [
    (0x0370, 0x03FF),  # Greek
    (0x0400, 0x052F),  # Cyrillic
    (0x0590, 0x05FF),  # Hebrew
    (0x0600, 0x06FF),  # Arabic
    (0x0750, 0x077F),
    (0x0900, 0x097F),  # Devanagari
    (0x0980, 0x09FF),  # Bengali
    (0x0E00, 0x0E7F),  # Thai
    (0x3040, 0x30FF),  # Japanese
    (0x3400, 0x4DBF),
    (0x4E00, 0x9FFF),  # CJK
    (0xAC00, 0xD7AF),  # Korean
]

ISSUE_PATTERNS = {
    "issue_crash_bug_flag": r"\b(?:crash(?:ed|es|ing)?|bug(?:s|gy)?|glitch(?:es|y)?|error(?:s)?|broken|not\s+working|doesn['’]?t\s+work|stopped\s+working|freeze(?:s|ing)?|frozen)\b",
    "issue_performance_loading_flag": r"\b(?:slow|slower|lag|lags|laggy|lagging|load(?:ing|s|ed)?|buffer(?:ing|s|ed)?|stuck|hang(?:s|ing)?|takes?\s+forever|performance)\b",
    "issue_login_account_flag": r"\b(?:log\s*in|login|sign\s*in|account|password|verification|verify|locked\s*out|suspend(?:ed|s|ing)?|ban(?:ned|s|ning)?)\b",
    "issue_payment_billing_flag": r"\b(?:payment|billing|bill|charged?|subscription|refund|price|expensive|cost|paywall|purchase|premium)\b",
    "issue_ads_flag": r"\b(?:ad|ads|advert(?:isement|isements|ising)?|commercials?|sponsored|promotion)\b",
    "issue_update_version_flag": r"\b(?:update|updated|updates|updating|version|downgrade|latest\s+update)\b",
    "issue_support_service_flag": r"\b(?:customer\s+service|support|help\s+center|contact\s+support|response|reply|agent|customer\s+care)\b",
}


def is_non_latin_letter(character):
    if not character.isalpha():
        return False
    code_point = ord(character)
    return any(start <= code_point <= end for start, end in NON_LATIN_RANGES)


def classify_language_v0(text):
    normalized = str(text or "").strip().lower()
    tokens = [token.lower() for token in WORD_RE.findall(normalized)]
    alphabetic_chars = [char for char in normalized if char.isalpha()]

    if len(alphabetic_chars) < 10 or len(tokens) < 3:
        return "undetermined", pd.NA, "too_short_for_rule"

    non_latin_count = sum(is_non_latin_letter(char) for char in alphabetic_chars)
    if non_latin_count >= 3 and non_latin_count / len(alphabetic_chars) >= 0.15:
        return "non_english_likely", 1, "non_latin_script"

    english_hits = sum(token in ENGLISH_MARKERS for token in tokens)
    language_scores = {
        language: sum(token in markers for token in tokens)
        for language, markers in LANGUAGE_MARKERS.items()
    }
    best_language = max(language_scores, key=language_scores.get)
    best_hits = language_scores[best_language]
    required_hits = 3 if best_language in {"tl", "hi_roman"} else 2

    if (
        best_hits >= required_hits
        and best_hits > english_hits
        and best_hits / len(tokens) >= 0.08
    ):
        return "non_english_likely", 1, f"{best_language}_markers"

    if english_hits >= 2 or english_hits / len(tokens) >= 0.15:
        return "english_likely", 0, "english_markers"

    return "undetermined", pd.NA, "insufficient_language_evidence"

print("Feature rules defined.")
print("Issue categories:", len(ISSUE_PATTERNS))


Feature rules defined.
Issue categories: 7


## 7. Generate review-level features

The output keeps stable review identifiers and cleaned text so later EDA can trace every derived value back to the validated database row.


In [7]:
features_df = reviews_df.copy()
text_series = features_df["content_cleaned"].fillna("").astype(str)

features_df["review_char_count"] = text_series.str.len().astype("int64")
features_df["review_word_count"] = text_series.map(
    lambda value: len(WORD_RE.findall(value))
).astype("int64")
features_df["alphanumeric_char_count"] = text_series.map(
    lambda value: sum(character.isalnum() for character in value)
).astype("int64")

features_df["short_review_flag"] = (
    features_df["review_char_count"] < SHORT_REVIEW_CHAR_THRESHOLD
).astype("int8")
features_df["low_signal_flag"] = (
    (features_df["review_word_count"] <= LOW_SIGNAL_MAX_WORDS)
    | (features_df["alphanumeric_char_count"] < LOW_SIGNAL_MIN_ALNUM_CHARS)
).astype("int8")

features_df["rating_group"] = pd.cut(
    features_df["score"],
    bins=[0, 2, 3, 5],
    labels=["low", "middle", "high"],
).astype("string")
features_df["low_rating_flag"] = features_df["score"].le(2).astype("int8")

features_df["duplicate_identity_flag"] = features_df.duplicated(
    ["source", "app_id", "review_id"],
    keep=False,
).astype("int8")

normalized_text = (
    text_series.str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
features_df["same_app_text_frequency"] = normalized_text.groupby(
    [features_df["app_id"], normalized_text]
).transform("size").astype("int64")
features_df["repeated_text_flag"] = features_df[
    "same_app_text_frequency"
].gt(1).astype("int8")

language_results = text_series.map(classify_language_v0)
features_df[[
    "language_group",
    "non_english_flag",
    "language_detection_reason",
]] = pd.DataFrame(language_results.tolist(), index=features_df.index)
features_df["non_english_flag"] = features_df["non_english_flag"].astype("Int64")

review_timestamp = pd.to_datetime(
    features_df["review_created_at"], utc=True, errors="coerce"
)
collection_timestamp = pd.to_datetime(
    features_df["fetched_at"], utc=True, errors="coerce"
)
features_df["review_date"] = review_timestamp.dt.strftime("%Y-%m-%d")
features_df["collection_date"] = collection_timestamp.dt.strftime("%Y-%m-%d")
features_df["review_age_hours_at_collection"] = (
    (collection_timestamp - review_timestamp).dt.total_seconds() / 3600
).round(3)

for feature_name, pattern in ISSUE_PATTERNS.items():
    features_df[feature_name] = text_series.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False,
    ).astype("int8")

issue_flag_columns = list(ISSUE_PATTERNS)
features_df["issue_indicator_count"] = features_df[
    issue_flag_columns
].sum(axis=1).astype("int8")
features_df["any_issue_indicator_flag"] = features_df[
    "issue_indicator_count"
].gt(0).astype("int8")

feature_column_order = [
    "review_key",
    "source",
    "app_id",
    "app_name",
    "review_id",
    "content_cleaned",
    "score",
    "rating_group",
    "low_rating_flag",
    "review_char_count",
    "review_word_count",
    "alphanumeric_char_count",
    "short_review_flag",
    "low_signal_flag",
    "duplicate_identity_flag",
    "same_app_text_frequency",
    "repeated_text_flag",
    "language_group",
    "non_english_flag",
    "language_detection_reason",
    "has_developer_reply",
    "review_created_at",
    "review_date",
    "fetched_at",
    "collection_date",
    "review_age_hours_at_collection",
    "app_version",
    "run_id",
    *issue_flag_columns,
    "issue_indicator_count",
    "any_issue_indicator_flag",
]
features_df = features_df[feature_column_order].copy()

print("Review-level features generated:", f"{len(features_df):,}")
print("Feature columns:", len(features_df.columns))
display(features_df.head(10))


Review-level features generated: 34,601
Feature columns: 37


,review_key,source,app_id,app_name,review_id,content_cleaned,score,rating_group,low_rating_flag,review_char_count,review_word_count,alphanumeric_char_count,short_review_flag,low_signal_flag,duplicate_identity_flag,same_app_text_frequency,repeated_text_flag,language_group,non_english_flag,language_detection_reason,has_developer_reply,review_created_at,review_date,fetched_at,collection_date,review_age_hours_at_collection,app_version,run_id,issue_crash_bug_flag,issue_performance_loading_flag,issue_login_account_flag,issue_payment_billing_flag,issue_ads_flag,issue_update_version_flag,issue_support_service_flag,issue_indicator_count,any_issue_indicator_flag
0,8700e0bccfc44936acc518dab7ffa6ce9aae9dfe5cd14b3cbdb3107ead5a56f6,google_play,com.dd.doordash,DoorDash,0aaf6b5c-9619-4576-8152-fac88d477570,They now charge for a Regular delivery 2.99 or wait on a 30 minute delivery for free cold food at a High price.. uni...,1,low,1,125,25,99,0,0,0,1,0,english_likely,0,english_markers,0,2026-06-28T23:28:16+00:00,2026-06-28,2026-07-08T03:44:59.159630+00:00,2026-07-08,220.279,15.281.1,phase2_day1_controlled_scale_20260708_034445,0,0,0,1,0,0,0,1,1
1,f40e0f6bd214601508a541dfbf7e6039d7857433844f8341ea1ac9d846fec414,google_play,com.dd.doordash,DoorDash,659e7426-23ec-4e60-847a-a337580a9494,Update: This app STILL deserves 1 star Loved this app! But I had a 40% off promo code that disappeared. The whole pr...,1,low,1,415,72,324,0,0,0,1,0,english_likely,0,english_markers,0,2026-06-28T23:30:27+00:00,2026-06-28,2026-07-08T03:44:59.159630+00:00,2026-07-08,220.242,15.281.1,phase2_day1_controlled_scale_20260708_034445,0,0,1,0,0,1,1,3,1
2,0774835d100ffc9bfa3b14e1f6195708721493fcb3010cfdf731dd47d63fd252,google_play,com.dd.doordash,DoorDash,4e435abe-e4a4-4e1e-ab87-0a0cf9d93d24,"Over priced app that always crashes. Use any other app, save your time and money.",1,low,1,81,15,64,0,0,0,1,0,english_likely,0,english_markers,0,2026-06-28T23:30:42+00:00,2026-06-28,2026-07-08T03:44:59.159630+00:00,2026-07-08,220.238,15.187.9,phase2_day1_controlled_scale_20260708_034445,1,0,0,0,0,0,0,1,1
3,02e837ed23ecc2c4cb4eb0e862fb856e7743df6bcdca5b210f6fe29cdd1fd841,google_play,com.dd.doordash,DoorDash,4f96af3b-e555-411e-9353-33b813fe1cfc,I keep getting an error message and I getting pissed off,1,low,1,56,11,46,0,0,0,1,0,english_likely,0,english_markers,0,2026-06-28T23:34:50+00:00,2026-06-28,2026-07-08T03:44:59.159630+00:00,2026-07-08,220.169,15.276.7,phase2_day1_controlled_scale_20260708_034445,1,0,0,0,0,0,0,1,1
4,660e98ecca88ba5086075447c3122900c6c22b3eb57d6befcba973cf9c4da5e4,google_play,com.dd.doordash,DoorDash,0bdc7e18-70d3-440d-8b23-7dc3c803cd64,trash,1,low,1,5,1,5,1,1,0,2,1,undetermined,<NA>,too_short_for_rule,0,2026-06-28T23:34:57+00:00,2026-06-28,2026-07-08T03:44:59.159630+00:00,2026-07-08,220.167,15.270.0,phase2_day1_controlled_scale_20260708_034445,0,0,0,0,0,0,0,0,0
5,2576a0aa2ae22c40b378b8d7b41af992423a51079f9b2a278cbc7a2ec3f0bce0,google_play,com.dd.doordash,DoorDash,ef59db6c-91eb-4521-8e16-b4901e9d8446,yaaaaaaaaaaaaaaaa,5,high,0,17,1,17,1,1,0,1,0,undetermined,<NA>,too_short_for_rule,0,2026-06-28T23:35:16+00:00,2026-06-28,2026-07-08T03:44:59.159630+00:00,2026-07-08,220.162,15.281.1,phase2_day1_controlled_scale_20260708_034445,0,0,0,0,0,0,0,0,0
6,70c9af4946861d1250c771259e6fa7c2901e78b6c722644d9ad43c2e38eddda1,google_play,com.dd.doordash,DoorDash,569bbdca-73c5-43e3-bbd9-b8109b9a6229,Spams notifications even if you turn off the marketing notifications in the app settings. And the Android-level noti...,1,low,1,393,68,315,0,0,0,1,0,english_likely,0,english_markers,0,2026-06-28T23:35:43+00:00,2026-06-28,2026-07-08T03:44:59.159630+00:00,2026-07-08,220.154,15.281.1,phase2_day1_controlled_scale_20260708_034445,0,0,0,0,0,0,0,0,0
7,0b9ab4b499bfddc75515f5d4c085e495cdb3ff97d7084edd759b5527682a9452,google_play,com.dd.doordash,DoorDash,14826d1b-7f45-4fb9-a115-0409e830b0c0,this app sucks. always erroring out and not allowing me to order.,1,low,1,65,12,52,0,0,0,1,0,english_likely,0,engli

## 8. Validate the review-level feature table

The checks below verify row preservation, identity uniqueness, score grouping, text lengths, timestamps, flag domains, and issue-count reconciliation.


In [8]:
binary_flag_columns = [
    "low_rating_flag",
    "short_review_flag",
    "low_signal_flag",
    "duplicate_identity_flag",
    "repeated_text_flag",
    "has_developer_reply",
    *issue_flag_columns,
    "any_issue_indicator_flag",
]

binary_domains_valid = all(
    set(features_df[column].dropna().unique()).issubset({0, 1})
    for column in binary_flag_columns
)
non_english_domain_valid = set(
    features_df["non_english_flag"].dropna().astype(int).unique()
).issubset({0, 1})

validation_rows = [
    ("feature_rows_match_cleaned_rows", len(features_df) == cleaned_rows, len(features_df)),
    ("review_key_complete", features_df["review_key"].notna().all(), int(features_df["review_key"].isna().sum())),
    ("review_key_unique", features_df["review_key"].is_unique, int(features_df["review_key"].duplicated().sum())),
    ("all_10_apps_present", features_df["app_id"].nunique() == EXPECTED_APP_COUNT, int(features_df["app_id"].nunique())),
    ("score_complete", features_df["score"].notna().all(), int(features_df["score"].isna().sum())),
    ("score_in_1_to_5", features_df["score"].isin([1, 2, 3, 4, 5]).all(), int((~features_df["score"].isin([1, 2, 3, 4, 5])).sum())),
    ("rating_group_complete", features_df["rating_group"].notna().all(), int(features_df["rating_group"].isna().sum())),
    ("rating_group_valid", features_df["rating_group"].isin(["low", "middle", "high"]).all(), int((~features_df["rating_group"].isin(["low", "middle", "high"])).sum())),
    ("content_length_matches", features_df["review_char_count"].eq(features_df["content_cleaned"].str.len()).all(), int((features_df["review_char_count"] != features_df["content_cleaned"].str.len()).sum())),
    ("nonnegative_word_counts", features_df["review_word_count"].ge(0).all(), int(features_df["review_word_count"].lt(0).sum())),
    ("review_timestamp_parseable", pd.to_datetime(features_df["review_created_at"], utc=True, errors="coerce").notna().all(), int(pd.to_datetime(features_df["review_created_at"], utc=True, errors="coerce").isna().sum())),
    ("collection_timestamp_parseable", pd.to_datetime(features_df["fetched_at"], utc=True, errors="coerce").notna().all(), int(pd.to_datetime(features_df["fetched_at"], utc=True, errors="coerce").isna().sum())),
    ("review_age_nonnegative", features_df["review_age_hours_at_collection"].ge(0).all(), int(features_df["review_age_hours_at_collection"].lt(0).sum())),
    ("binary_flag_domains_valid", binary_domains_valid, len(binary_flag_columns)),
    ("non_english_flag_domain_valid", non_english_domain_valid, int(features_df["non_english_flag"].notna().sum())),
    ("duplicate_identity_flags_zero_after_dedup", features_df["duplicate_identity_flag"].eq(0).all(), int(features_df["duplicate_identity_flag"].sum())),
    ("issue_count_reconciles", features_df["issue_indicator_count"].eq(features_df[issue_flag_columns].sum(axis=1)).all(), int((features_df["issue_indicator_count"] != features_df[issue_flag_columns].sum(axis=1)).sum())),
    ("any_issue_flag_reconciles", features_df["any_issue_indicator_flag"].eq(features_df["issue_indicator_count"].gt(0).astype(int)).all(), int((features_df["any_issue_indicator_flag"] != features_df["issue_indicator_count"].gt(0).astype(int)).sum())),
]
validation_df = pd.DataFrame(
    validation_rows,
    columns=["validation_check", "passed", "observed_value"],
)

if not validation_df["passed"].all():
    raise ValueError(
        "Feature validation failed:\n"
        + validation_df.loc[~validation_df["passed"]].to_string(index=False)
    )

print("Review-level validation checks:", len(validation_df))
print("Failed checks:", int((~validation_df["passed"]).sum()))
display(validation_df)


Review-level validation checks: 18
Failed checks: 0


,validation_check,passed,observed_value
0,feature_rows_match_cleaned_rows,True,34601
1,review_key_complete,True,0
2,review_key_unique,True,0
3,all_10_apps_present,True,10
4,score_complete,True,0
5,score_in_1_to_5,True,0
6,rating_group_complete,True,0
7,rating_group_valid,True,0
8,content_length_matches,True,0
9,nonnegative_word_counts,True,0


## 9. Create app-level aggregate features

The aggregates summarize volume, rating distribution, text signal, language status, developer replies, and issue indicators for each app.


In [9]:
def non_english_all_share(series):
    return float(series.eq(1).mean())


def non_english_classified_share(series):
    classified = series.dropna()
    if classified.empty:
        return np.nan
    return float(classified.eq(1).mean())

aggregate_spec = {
    "review_volume": ("review_key", "size"),
    "average_rating": ("score", "mean"),
    "low_rating_share": ("low_rating_flag", "mean"),
    "middle_rating_share": ("rating_group", lambda values: values.eq("middle").mean()),
    "high_rating_share": ("rating_group", lambda values: values.eq("high").mean()),
    "average_review_char_count": ("review_char_count", "mean"),
    "median_review_char_count": ("review_char_count", "median"),
    "average_review_word_count": ("review_word_count", "mean"),
    "short_review_share": ("short_review_flag", "mean"),
    "low_signal_share": ("low_signal_flag", "mean"),
    "repeated_text_share": ("repeated_text_flag", "mean"),
    "developer_reply_share": ("has_developer_reply", "mean"),
    "non_english_share_all_reviews": ("non_english_flag", non_english_all_share),
    "non_english_share_classified": ("non_english_flag", non_english_classified_share),
    "language_undetermined_share": ("language_group", lambda values: values.eq("undetermined").mean()),
    "average_review_age_hours_at_collection": ("review_age_hours_at_collection", "mean"),
    "any_issue_indicator_share": ("any_issue_indicator_flag", "mean"),
}

for issue_column in issue_flag_columns:
    aggregate_spec[issue_column.replace("_flag", "_share")] = (
        issue_column,
        "mean",
    )

app_aggregates_df = (
    features_df.groupby(["app_name", "app_id", "source"], as_index=False)
    .agg(**aggregate_spec)
    .sort_values("app_name")
    .reset_index(drop=True)
)

share_columns = [
    column for column in app_aggregates_df.columns
    if column.endswith("_share") or "_share_" in column
]
app_aggregates_df[share_columns] = app_aggregates_df[share_columns].round(6)
for column in [
    "average_rating",
    "average_review_char_count",
    "median_review_char_count",
    "average_review_word_count",
    "average_review_age_hours_at_collection",
]:
    app_aggregates_df[column] = app_aggregates_df[column].round(3)

aggregate_validation_rows = [
    ("ten_app_aggregate_rows", len(app_aggregates_df) == EXPECTED_APP_COUNT, len(app_aggregates_df)),
    ("aggregate_volume_matches_review_rows", int(app_aggregates_df["review_volume"].sum()) == len(features_df), int(app_aggregates_df["review_volume"].sum())),
    ("rating_shares_sum_to_one", np.allclose(app_aggregates_df[["low_rating_share", "middle_rating_share", "high_rating_share"]].sum(axis=1), 1.0, atol=1e-6), float((app_aggregates_df[["low_rating_share", "middle_rating_share", "high_rating_share"]].sum(axis=1) - 1).abs().max())),
    ("weighted_average_rating_matches", np.isclose(np.average(app_aggregates_df["average_rating"], weights=app_aggregates_df["review_volume"]), features_df["score"].mean(), atol=0.001), float(features_df["score"].mean())),
]
aggregate_validation_df = pd.DataFrame(
    aggregate_validation_rows,
    columns=["validation_check", "passed", "observed_value"],
)
if not aggregate_validation_df["passed"].all():
    raise ValueError("App-level aggregate validation failed.")

print("App aggregate rows:", len(app_aggregates_df))
display(app_aggregates_df)


App aggregate rows: 10


,app_name,app_id,source,review_volume,average_rating,low_rating_share,middle_rating_share,high_rating_share,average_review_char_count,median_review_char_count,average_review_word_count,short_review_share,low_signal_share,repeated_text_share,developer_reply_share,non_english_share_all_reviews,non_english_share_classified,language_undetermined_share,average_review_age_hours_at_collection,any_issue_indicator_share,issue_crash_bug_share,issue_performance_loading_share,issue_login_account_share,issue_payment_billing_share,issue_ads_share,issue_update_version_share,issue_support_service_share
0,DoorDash,com.dd.doordash,google_play,1975,3.101,0.442025,0.063291,0.494684,114.070,59.0,21.291,0.234937,0.168101,0.089114,0.000000,0.002495,0.002495,0.188354,100.816,0.274430,0.033924,0.013165,0.032911,0.162025,0.007089,0.024810,0.077468
1,Duolingo,com.duolingo,google_play,2098,4.500,0.076263,0.045281,0.878456,61.697,28.0,11.730,0.396568,0.304576,0.186368,0.000000,0.041785,0.041785,0.326978,37.809,0.056721,0.010963,0.005720,0.002860,0.011916,0.017636,0.013346,0.005243
2,Google Maps,com.google.android.apps.maps,google_play,2635,3.614,0.316129,0.056926,0.626945,84.824,24.0,15.514,0.462619,0.403036,0.266034,0.210626,0.037685,0.037685,0.436053,81.314,0.137002,0.036053,0.022391,0.004175,0.002657,0.007590,0.076660,0.007211
3,Instagram,com.instagram.android,google_play,6045,3.847,0.262200,0.035070,0.702730,55.742,18.0,10.265,0.525558,0.443507,0.279074,0.000000,0.072235,0.072235,0.487014,30.950,0.184781,0.029942,0.008768,0.132341,0.002481,0.009595,0.022994,0.014888
4,Netflix,com.netflix.mediaclient,google_play,2365,3.519,0.336575,0.050740,0.612685,86.827,40.0,15.819,0.343340,0.288372,0.167865,0.000000,0.026201,0.026201,0.322199,99.416,0.198309,0.040592,0.024524,0.047357,0.082452,0.022410,0.028753,0.012262
5,Reddit,com.reddit.frontpage,google_play,1991,2.984,0.483677,0.028629,0.487695,96.926,51.0,17.918,0.309392,0.239076,0.142140,0.000000,0.003394,0.003394,0.260171,110.131,0.201406,0.033651,0.019086,0.104972,0.004520,0.030638,0.043697,0.009041
6,Spotify,com.spotify.music,google_play,4174,3.883,0.231433,0.065165,0.703402,77.409,34.0,14.696,0.350264,0.268328,0.166028,0.108529,0.012150,0.012150,0.290129,39.246,0.265932,0.026833,0.025635,0.011979,0.125539,0.112123,0.035458,0.005510
7,TikTok,com.zhiliaoapp.musically,google_play,4059,3.839,0.257699,0.056171,0.686130,57.209,20.0,10.970,0.491993,0.393447,0.217295,0.842079,0.040137,0.040137,0.423011,41.821,0.160138,0.029810,0.026115,0.070214,0.004681,0.016999,0.026854,0.012811
8,Uber,com.ubercab,google_play,3440,3.839,0.270058,0.028198,0.701744,70.508,21.0,13.025,0.482849,0.428198,0.328779,0.002326,0.035771,0.035771,0.455523,53.251,0.131686,0.010756,0.006105,0.009884,0.081395,0.002907,0.004360,0.043023
9,YouTube,com.google.android.youtube,google_play,5819,3.660,0.308816,0.046915,0.644269,54.629,15.0,10.284,0.572779,0.486166,0.278226,0.000000,0.043494,0.043494,0.521911,32.507,0.167211,0.038323,0.018560,0.009280,0.020107,0.067881,0.043994,0.006874


## 10. Build a small, deterministic sample output

For each app, the sample includes a newest review, an issue-indicator example, and a low-signal example when distinct rows are available. Review text is limited to a short preview and user names are not exported.


In [10]:
def choose_sample_rows(app_frame):
    chosen = []
    used_keys = set()

    candidates = [
        (
            "newest_review",
            app_frame.sort_values(
                ["review_created_at", "review_key"],
                ascending=[False, True],
            ),
        ),
        (
            "issue_example",
            app_frame.loc[app_frame["any_issue_indicator_flag"].eq(1)].sort_values(
                ["low_rating_flag", "review_created_at", "review_key"],
                ascending=[False, False, True],
            ),
        ),
        (
            "low_signal_example",
            app_frame.loc[app_frame["low_signal_flag"].eq(1)].sort_values(
                ["review_created_at", "review_key"],
                ascending=[False, True],
            ),
        ),
    ]

    for sample_type, candidate_frame in candidates:
        available = candidate_frame.loc[
            ~candidate_frame["review_key"].isin(used_keys)
        ]
        if available.empty:
            available = app_frame.loc[~app_frame["review_key"].isin(used_keys)]
        if available.empty:
            continue
        row = available.iloc[0].copy()
        row["sample_type"] = sample_type
        used_keys.add(row["review_key"])
        chosen.append(row)

    return pd.DataFrame(chosen)

sample_df = pd.concat(
    [
        choose_sample_rows(app_frame)
        for _, app_frame in features_df.groupby("app_name", sort=True)
    ],
    ignore_index=True,
)

sample_df["content_preview"] = (
    sample_df["content_cleaned"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 160)
)

sample_columns = [
    "sample_type",
    "review_key",
    "app_name",
    "app_id",
    "score",
    "rating_group",
    "review_char_count",
    "review_word_count",
    "short_review_flag",
    "low_signal_flag",
    "duplicate_identity_flag",
    "repeated_text_flag",
    "language_group",
    "non_english_flag",
    "has_developer_reply",
    "review_date",
    "collection_date",
    "review_age_hours_at_collection",
    *issue_flag_columns,
    "issue_indicator_count",
    "content_preview",
]
sample_df = sample_df[sample_columns].sort_values(
    ["app_name", "sample_type"]
).reset_index(drop=True)

if len(sample_df) != EXPECTED_APP_COUNT * 3:
    raise ValueError("The sample output should contain three rows per app.")

print("Sample rows:", len(sample_df))
display(sample_df)


Sample rows: 30


,sample_type,review_key,app_name,app_id,score,rating_group,review_char_count,review_word_count,short_review_flag,low_signal_flag,duplicate_identity_flag,repeated_text_flag,language_group,non_english_flag,has_developer_reply,review_date,collection_date,review_age_hours_at_collection,issue_crash_bug_flag,issue_performance_loading_flag,issue_login_account_flag,issue_payment_billing_flag,issue_ads_flag,issue_update_version_flag,issue_support_service_flag,issue_indicator_count,content_preview
0,issue_example,1dbd0a653eebf065622a8b8a1e433a623074bc4c99fef15735b909906f9adee8,DoorDash,com.dd.doordash,1,low,13,2,1,1,0,0,undetermined,<NA>,0,2026-07-13,2026-07-14,24.840,0,0,0,1,0,0,0,1,roo expensive
1,low_signal_example,eae41fad48108f9217729447ad7d8b1315e8d0857f3d3c6fde0ee9af2f7a84da,DoorDash,com.dd.doordash,5,high,7,1,1,1,0,1,undetermined,<NA>,0,2026-07-13,2026-07-14,26.719,0,0,0,0,0,0,0,0,amazing
2,newest_review,5595a71d59741c1a254f87c10a2e11ab22380a7d0f570752e104cbd59ba0a688,DoorDash,com.dd.doordash,5,high,14,4,1,0,0,0,english_likely,0,0,2026-07-13,2026-07-14,24.072,0,0,0,0,0,0,0,0,I love it here
3,issue_example,9ce3095b6448d7a4a81909660c0b7bb37c36d163f5eaa3ceb04c943d8146d66a,Duolingo,com.duolingo,1,low,25,6,0,0,0,0,english_likely,0,0,2026-07-13,2026-07-14,25.223,0,0,0,0,1,0,0,1,go to hell with those ads
4,low_signal_example,bc759ac7d36b8601180ac80fd729a70da7fce659537a329b37baafb818e1c244,Duolingo,com.duolingo,5,high,8,1,1,1,0,0,undetermined,<NA>,0,2026-07-13,2026-07-14,31.849,0,0,0,0,0,0,0,0,Duolingo
5,newest_review,19a1b6dc412c7ef63399973b82e00999293f68454c3381aeef57391ae0ef97ca,Duolingo,com.duolingo,5,high,14,2,1,1,0,0,undetermined,<NA>,0,2026-07-13,2026-07-14,24.487,0,0,0,0,0,0,0,0,love it 😀 😍 ❤️
6,issue_example,b0ce21fbcab618811a93ccfbf27af07ccec7bdedd1302863efcc7ac5a987f241,Google Maps,com.google.android.apps.maps,1,low,46,8,0,0,0,0,english_likely,0,0,2026-07-13,2026-07-14,24.307,0,0,0,0,0,1,0,1,huge battery drain after the Android 17 update
7,low_signal_example,f27922beeca7ed530ce0e9bcc1cdbef40f96413484bf894fe1ca1604c4d06c26,Google Maps,com.google.android.apps.maps,5,high,7,1,1,1,0,0,undetermined,<NA>,0,2026-07-13,2026-07-14,24.536,0,0,0,0,0,0,0,0,super 💕
8,newest_review,676d375efad48e6d7e186b6cffda8060966ea8f0a0b5ad6ce06d29a173599f61,Google Maps,com.google.android.apps.maps,1,low,81,15,0,0,0,0,non_english_likely,1,0,2026-07-13,2026-07-14,24.041,0,0,0,0,0,0,0,0,بياخد وقت كبير عندي علشان يحدد المكان والمسار ياريت تخديث يحل المشكله دي oppo A78
9,issue_example,58679eed63212cd7efab1de5a9172cd6687f6f69977b7b84cf8d22b42e5dd5ff,Instagram,com.instagram.android,1,low,227,37,0,0,0,0,english_likely,0,0,2026-07-13,2026-07-14,24.016,0,0,1,0,0,0,0,1,"Hello Instagram Team, my account has no Community Standards violations in Account Status, but I cannot share Reels. ..."


## 11. Document feature definitions and reliability

The dictionary records what each field means, how it is calculated, its later use, and its main limitation. The reliability table separates deterministic/source-provided features from heuristic features.


In [11]:
feature_definition_rows = [
    ("review_key", "traceability", "Stable hashed review key from the database.", "Loaded from phase2_reviews_raw.review_key.", "Join features back to raw and cleaned review records.", "High; validated unique."),
    ("source", "traceability", "Review source platform.", "Loaded from the raw review table.", "Source filtering and future multi-source support.", "High."),
    ("app_id", "app identity", "Google Play package identifier.", "Loaded from the raw review table.", "Stable app grouping and joins.", "High."),
    ("app_name", "app identity", "Human-readable app name.", "Loaded from the raw review table.", "EDA labels and app-level summaries.", "High; depends on maintained app metadata."),
    ("review_id", "traceability", "Source review identifier.", "Loaded from the raw review table.", "Deduplication and review tracing.", "High; complete in this snapshot."),
    ("content_cleaned", "text", "Cleaned review text retained for later analysis.", "Loaded from phase2_reviews_cleaned.content_cleaned.", "Later EDA, sentiment, topic, and issue analysis.", "High as stored text; cleaning may remove source formatting."),
    ("score", "rating", "Source-provided 1–5 review rating.", "Loaded from the cleaned review table.", "Rating summaries and later target/validation use.", "High; complete and valid in this snapshot."),
    ("rating_group", "rating", "Low, middle, or high rating band.", "1–2=low; 3=middle; 4–5=high.", "Simple rating segmentation and app comparisons.", "High; deterministic, but grouping is project-defined."),
    ("low_rating_flag", "rating", "Indicates a rating of 1 or 2.", "1 when score <= 2, otherwise 0.", "Low-rating share and issue screening.", "High; deterministic."),
    ("review_char_count", "text length", "Number of characters in cleaned review text.", "Python string length after stored cleaning.", "Text-length distribution and short-review checks.", "High; deterministic."),
    ("review_word_count", "text length", "Number of Unicode word tokens.", "Count regex word tokens in cleaned text.", "Text-length EDA and low-signal screening.", "Medium-high; tokenization is simple and language-neutral, not linguistic parsing."),
    ("alphanumeric_char_count", "text signal", "Number of alphabetic or numeric characters.", "Count characters where isalnum() is true.", "Distinguish textual content from emoji/punctuation-only content.", "High; deterministic."),
    ("short_review_flag", "text signal", "Review is shorter than the v0 character threshold.", f"1 when review_char_count < {SHORT_REVIEW_CHAR_THRESHOLD}.", "Short-review share and EDA filtering.", "Medium; threshold is transparent but project-defined."),
    ("low_signal_flag", "text signal", "Review has very little usable text under the v0 rule.", f"1 when review_word_count <= {LOW_SIGNAL_MAX_WORDS} or alphanumeric_char_count < {LOW_SIGNAL_MIN_ALNUM_CHARS}.", "Identify rows needing caution in text analysis.", "Medium; a short review can still be meaningful."),
    ("duplicate_identity_flag", "deduplication", "Repeated source/app/review identity in the feature snapshot.", "Duplicate check on source + app_id + review_id.", "Validate deduplication before downstream use.", "High, but all values are 0 after database deduplication."),
    ("same_app_text_frequency", "text repetition", "Count of identical normalized cleaned text within the same app.", "Group lowercased whitespace-normalized text by app.", "Identify generic/repeated text without confusing it with identity duplication.", "Medium; identical text may still come from different legitimate users."),
    ("repeated_text_flag", "text repetition", "Same normalized text appears more than once within an app.", "1 when same_app_text_frequency > 1.", "Repeated/generic text screening.", "Medium; not a duplicate-review identity."),
    ("language_group", "language heuristic", "english_likely, non_english_likely, or undetermined.", "Conservative script and marker-word rule.", "Language-aware EDA and text-routing decisions.", "Low-medium; not a trained language detector."),
    ("non_english_flag", "language heuristic", "Likely non-English text indicator.", "1 for non_english_likely, 0 for english_likely, blank for undetermined.", "Estimate non-English coverage while preserving uncertainty.", "Low-medium; short and mixed-language reviews are difficult."),
    ("language_detection_reason", "language heuristic", "Reason assigned by the v0 language rule.", "Records script, marker, short-text, or insufficient-evidence path.", "Audit heuristic decisions and future replacement.", "Medium as a rule trace; not language truth."),
    ("has_developer_reply", "developer reply", "Whether a developer reply is present.", "Loaded from the cleaned review table.", "Measure response availability and support analysis.", "High for presence; does not measure reply quality or timeliness."),
    ("review_created_at", "time", "Source review timestamp.", "Loaded from the database.", "Temporal EDA and freshness analysis.", "High as stored; source returned-window behavior still applies."),
    ("review_date", "time", "UTC review calendar date.", "Date extracted from review_created_at.", "Daily review volume and rating trends.", "High; deterministic UTC conversion."),
    ("fetched_at", "time", "Collection timestamp for the first stored review identity.", "Loaded from the raw review table.", "Collection traceability and source-lag analysis.", "High for first-seen collection time."),
    ("collection_date", "time", "UTC collection calendar date.", "Date extracted from fetched_at.", "Collection cohort summaries.", "High; deterministic UTC conversion."),
    ("review_age_hours_at_collection", "time", "Hours between review timestamp and first stored collection timestamp.", "(fetched_at - review_created_at) in hours.", "Returned-window lag and freshness diagnostics.", "High mathematically; interpretation depends on source behavior."),
    ("app_version", "source metadata", "App version reported with the review when available.", "Loaded from the raw review table.", "Version-specific issue analysis.", "Medium; naturally missing for many reviews."),
    ("run_id", "traceability", "Ingestion run that first inserted the review identity.", "Loaded from the raw review table.", "Cohort and run-level tracing.", "High."),
]

issue_definition_text = {
    "issue_crash_bug_flag": "Crash, bug, glitch, error, broken, freeze, or not-working terms.",
    "issue_performance_loading_flag": "Slow, lag, loading, buffering, stuck, hanging, or performance terms.",
    "issue_login_account_flag": "Login, account, password, verification, suspension, or ban terms.",
    "issue_payment_billing_flag": "Payment, billing, charge, subscription, refund, price, purchase, or premium terms.",
    "issue_ads_flag": "Ad, advertising, commercial, sponsored, or promotion terms.",
    "issue_update_version_flag": "Update, version, latest-update, or downgrade terms.",
    "issue_support_service_flag": "Support, customer service/care, help center, response, reply, or agent terms.",
}
for issue_name, meaning in issue_definition_text.items():
    feature_definition_rows.append((
        issue_name,
        "keyword issue indicator",
        meaning,
        "Case-insensitive regex keyword/phrase match in cleaned review text.",
        "Initial issue screening and app-level issue shares.",
        "Low-medium; keyword matches can miss context, spelling variants, negation, and non-English issues.",
    ))

feature_definition_rows.extend([
    ("issue_indicator_count", "keyword issue indicator", "Number of issue categories matched by a review.", "Sum of the seven issue flags.", "Prioritize multi-issue reviews for inspection.", "Low-medium; inherits keyword limitations."),
    ("any_issue_indicator_flag", "keyword issue indicator", "At least one issue category matched.", "1 when issue_indicator_count > 0.", "Basic issue-share aggregation.", "Low-medium; not a verified issue label."),
])

feature_definitions_df = pd.DataFrame(
    feature_definition_rows,
    columns=[
        "feature_name",
        "feature_group",
        "definition",
        "calculation",
        "possible_later_use",
        "reliability_and_limitation",
    ],
)

classified_language_rows = int(features_df["non_english_flag"].notna().sum())
quality_assessment_rows = [
    ("app and review identifiers", "most useful", "high", 1.0, "Complete and validated; support stable joins and grouping."),
    ("score and rating_group", "most useful", "high", 1.0, "Complete source rating plus a deterministic band for EDA."),
    ("review and collection timestamps", "most useful", "high", 1.0, "Complete and parseable; useful for cohorts and source-lag analysis."),
    ("review length features", "most useful", "high", 1.0, "Deterministic and directly useful for text-quality EDA."),
    ("has_developer_reply", "useful with caution", "high for presence", 1.0, "Reliable presence flag, but highly uneven across apps and not a reply-quality measure."),
    ("duplicate_identity_flag", "useful validation / low analytic variance", "high", 1.0, "All zeros in the validated deduplicated database; useful as a guard rather than a predictive feature."),
    ("repeated_text_flag", "useful with caution", "medium", 1.0, "Shows repeated content, but repeated text is not the same as duplicate review identity."),
    ("short_review_flag and low_signal_flag", "useful with caution", "medium", 1.0, "Transparent thresholds support screening, but short text can still be meaningful."),
    ("language_group and non_english_flag", "least reliable", "low-medium", classified_language_rows / len(features_df), "Conservative rule leaves short or unclear text undetermined; should be replaced by a validated language detector later."),
    ("keyword issue indicators", "least reliable", "low-medium", 1.0, "Useful for initial screening only; context, negation, spelling, and non-English wording can cause misses or false matches."),
]
feature_quality_df = pd.DataFrame(
    quality_assessment_rows,
    columns=[
        "feature_or_group",
        "assessment",
        "reliability",
        "coverage_share",
        "reason",
    ],
)
feature_quality_df["coverage_share"] = feature_quality_df["coverage_share"].round(6)

print("Feature definitions:", len(feature_definitions_df))
print("Reliability assessment rows:", len(feature_quality_df))
display(feature_quality_df)


Feature definitions: 37
Reliability assessment rows: 10


,feature_or_group,assessment,reliability,coverage_share,reason
0,app and review identifiers,most useful,high,1.000000,Complete and validated; support stable joins and grouping.
1,score and rating_group,most useful,high,1.000000,Complete source rating plus a deterministic band for EDA.
2,review and collection timestamps,most useful,high,1.000000,Complete and parseable; useful for cohorts and source-lag analysis.
3,review length features,most useful,high,1.000000,Deterministic and directly useful for text-quality EDA.
4,has_developer_reply,useful with caution,high for presence,1.000000,"Reliable presence flag, but highly uneven across apps and not a reply-quality measure."
5,duplicate_identity_flag,useful validation / low analytic variance,high,1.000000,All zeros in the validated deduplicated database; useful as a guard rather than a predictive feature.
6,repeated_text_flag,useful with caution,medium,1.000000,"Shows repeated content, but repeated text is not the same as duplicate review identity."
7,short_review_flag and low_signal_flag,useful with caution,medium,1.000000,"Transparent thresholds support screening, but short text can still be meaningful."
8,language_group and non_english_flag,least reliable,low-medium,0.596457,Conservative rule leaves short or unclear text undetermined; should be replaced by a validated language detector later.
9,keyword issue indicators,least reliable,low-medium,1.000000,"Useful for initial screening only; context, negation, spelling, and non-English wording can cause misses or false ma..."


## 12. Summarize actual v0 results

These results describe this validated 34,601-review snapshot only. Heuristic indicators are reported as screening signals, not ground-truth sentiment, language, topic, or issue labels.


In [12]:
overall_summary = {
    "review_rows": int(len(features_df)),
    "app_count": int(features_df["app_id"].nunique()),
    "average_rating": float(features_df["score"].mean()),
    "low_rating_share": float(features_df["low_rating_flag"].mean()),
    "short_review_share": float(features_df["short_review_flag"].mean()),
    "low_signal_share": float(features_df["low_signal_flag"].mean()),
    "repeated_text_share": float(features_df["repeated_text_flag"].mean()),
    "developer_reply_share": float(features_df["has_developer_reply"].mean()),
    "english_likely_share": float(features_df["language_group"].eq("english_likely").mean()),
    "non_english_likely_share": float(features_df["language_group"].eq("non_english_likely").mean()),
    "language_undetermined_share": float(features_df["language_group"].eq("undetermined").mean()),
    "non_english_share_classified": float(features_df.loc[features_df["non_english_flag"].notna(), "non_english_flag"].mean()),
    "any_issue_indicator_share": float(features_df["any_issue_indicator_flag"].mean()),
    "duplicate_identity_rows": int(features_df["duplicate_identity_flag"].sum()),
}

issue_rate_df = pd.DataFrame({
    "feature_name": issue_flag_columns,
    "review_count": [int(features_df[column].sum()) for column in issue_flag_columns],
    "review_share": [float(features_df[column].mean()) for column in issue_flag_columns],
}).sort_values("review_share", ascending=False)
issue_rate_df["review_share"] = issue_rate_df["review_share"].round(6)

summary_table_df = pd.DataFrame([
    ("Review rows", f"{overall_summary['review_rows']:,}"),
    ("Apps", overall_summary["app_count"]),
    ("Average rating", f"{overall_summary['average_rating']:.3f}"),
    ("Low-rating share", f"{overall_summary['low_rating_share']:.2%}"),
    ("Short-review share", f"{overall_summary['short_review_share']:.2%}"),
    ("Low-signal share", f"{overall_summary['low_signal_share']:.2%}"),
    ("Repeated-text share", f"{overall_summary['repeated_text_share']:.2%}"),
    ("Developer-reply share", f"{overall_summary['developer_reply_share']:.2%}"),
    ("Likely non-English share (all reviews)", f"{overall_summary['non_english_likely_share']:.2%}"),
    ("Language undetermined share", f"{overall_summary['language_undetermined_share']:.2%}"),
    ("Any keyword issue share", f"{overall_summary['any_issue_indicator_share']:.2%}"),
    ("Duplicate identity rows", overall_summary["duplicate_identity_rows"]),
], columns=["metric", "result"])

print("Overall feature summary:")
display(summary_table_df)
print("Keyword issue indicator rates:")
display(issue_rate_df)


Overall feature summary:


,metric,result
0,Review rows,"34,601"
1,Apps,10
2,Average rating,3.725
3,Low-rating share,28.75%
4,Short-review share,45.01%
5,Low-signal share,37.26%
6,Repeated-text share,23.01%
7,Developer-reply share,12.81%
8,Likely non-English share (all reviews),2.05%
9,Language undetermined share,40.35%


Keyword issue indicator rates:


,feature_name,review_count,review_share
2,issue_login_account_flag,1626,0.046993
3,issue_payment_billing_flag,1511,0.043669
4,issue_ads_flag,1185,0.034248
5,issue_update_version_flag,1101,0.031820
0,issue_crash_bug_flag,1022,0.029537
1,issue_performance_loading_flag,588,0.016994
6,issue_support_service_flag,583,0.016849


## 13. Export outputs and GitHub documentation

The report clearly separates deterministic/source-provided features from heuristic features and records the operational dependency on the monitoring layer.


In [13]:
review_features_path = OUTPUT_DIR / "review_features_v0.csv"
review_sample_path = OUTPUT_DIR / "review_feature_sample_v0.csv"
app_aggregates_path = OUTPUT_DIR / "app_feature_aggregates_v0.csv"
feature_definitions_path = OUTPUT_DIR / "feature_definitions_v0.csv"
feature_quality_path = OUTPUT_DIR / "feature_quality_assessment_v0.csv"
validation_path = OUTPUT_DIR / "feature_validation_checks_v0.csv"
source_validation_path = OUTPUT_DIR / "feature_engineering_source_validation_v0.csv"
issue_rates_path = OUTPUT_DIR / "feature_issue_indicator_rates_v0.csv"
metadata_output_path = OUTPUT_DIR / "feature_engineering_metadata_v0.json"
report_path = REPORT_DIR / "google_play_feature_engineering_v0_report.md"
dictionary_report_path = REPORT_DIR / "feature_engineering_v0_feature_dictionary.md"
readme_update_path = REPORT_DIR / "README_feature_engineering_v0_update.md"

features_df.to_csv(review_features_path, index=False)
sample_df.to_csv(review_sample_path, index=False)
app_aggregates_df.to_csv(app_aggregates_path, index=False)
feature_definitions_df.to_csv(feature_definitions_path, index=False)
feature_quality_df.to_csv(feature_quality_path, index=False)
pd.concat([source_guard_df, validation_df, aggregate_validation_df], ignore_index=True).to_csv(
    validation_path,
    index=False,
)
source_validation_df.to_csv(source_validation_path, index=False)
issue_rate_df.to_csv(issue_rates_path, index=False)


def markdown_table(frame, max_rows=None):
    view = frame.copy()
    if max_rows is not None:
        view = view.head(max_rows)
    view = view.fillna("").astype(str)
    header = "| " + " | ".join(view.columns) + " |"
    separator = "|" + "|".join(["---"] * len(view.columns)) + "|"
    rows = [
        "| " + " | ".join(value.replace("|", "/") for value in row) + " |"
        for row in view.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator] + rows)

app_report_view = app_aggregates_df[[
    "app_name",
    "review_volume",
    "average_rating",
    "low_rating_share",
    "short_review_share",
    "low_signal_share",
    "developer_reply_share",
    "any_issue_indicator_share",
]].copy()
for column in [
    "low_rating_share",
    "short_review_share",
    "low_signal_share",
    "developer_reply_share",
    "any_issue_indicator_share",
]:
    app_report_view[column] = app_report_view[column].map(lambda value: f"{value:.2%}")

issue_report_view = issue_rate_df.copy()
issue_report_view["review_share"] = issue_report_view["review_share"].map(
    lambda value: f"{value:.2%}"
)

report_text = f"""
# Google Play Review Feature Engineering v0 Report

## Scope

This feature layer is generated from the validated Phase 2 SQLite database containing {len(features_df):,} cleaned Google Play reviews across {features_df['app_id'].nunique()} apps.

The version is intentionally lightweight. It does not train a sentiment model, topic model, issue classifier, or any other machine-learning model. It defines transparent features that can later support EDA and model design.

During normal operations, the sequence remains:

1. run the normal ingestion process
2. run the monitoring layer
3. stop downstream use if a hard failure is present
4. document warnings and continue only when the run remains usable
5. generate or refresh the feature layer

## Review-level features

The output includes app identity, score and rating group, cleaned-text length, short and low-signal flags, review and collection dates, first-collection age, deduplication checks, repeated-text status, language heuristic status, developer-reply availability, and seven simple issue indicators.

The complete definitions are stored in `outputs/feature_definitions_v0.csv` and `reports/feature_engineering_v0_feature_dictionary.md`.

## Overall results

- Review rows: {len(features_df):,}
- Apps: {features_df['app_id'].nunique()}
- Average rating: {overall_summary['average_rating']:.3f}
- Low-rating reviews: {overall_summary['low_rating_share']:.2%}
- Short reviews: {overall_summary['short_review_share']:.2%}
- Low-signal reviews under the v0 rule: {overall_summary['low_signal_share']:.2%}
- Repeated cleaned text within the same app: {overall_summary['repeated_text_share']:.2%}
- Reviews with a developer reply: {overall_summary['developer_reply_share']:.2%}
- Likely non-English reviews: {overall_summary['non_english_likely_share']:.2%} of all reviews
- Language undetermined: {overall_summary['language_undetermined_share']:.2%}, mainly because many reviews are too short for a reliable rule
- Reviews matching at least one issue indicator: {overall_summary['any_issue_indicator_share']:.2%}
- Duplicate review identities in the persisted feature snapshot: {overall_summary['duplicate_identity_rows']}

## App-level aggregates

{markdown_table(app_report_view)}

The app-level values describe the collected database snapshot and should not be treated as general app quality rankings.

## Issue indicator rates

{markdown_table(issue_report_view)}

The issue indicators are keyword screens only. A match does not prove that the issue occurred, and a non-match does not prove that it did not occur.

## Most useful features

The strongest first-version features are app identity, score/rating group, review and collection timestamps, review-length measures, and developer-reply availability. They are source-provided or deterministic and can directly support EDA and data-quality segmentation.

The duplicate identity flag is also reliable, but it has no variance in this persisted database because the ingestion process already removed duplicate review identities. It is most useful as a validation guard.

## Least reliable features

The non-English flag, low-signal flag, and keyword issue indicators are heuristic. The language rule deliberately leaves uncertain and short reviews as `undetermined`. The keyword rules do not understand context, negation, spelling variation, or issue wording in languages outside the current English keyword groups.

These fields are appropriate for initial filtering and descriptive EDA. They should not be presented as ground-truth sentiment, language, topic, or issue labels.

## Validation

All source-package, database, review-level, and app-level validation checks passed. The feature output preserved all {len(features_df):,} cleaned review rows, retained 10 apps, created no duplicate identities, and reconciled all derived counts and app-level volumes.
"""
report_text = textwrap.dedent(report_text).strip() + "\n"
report_path.write_text(report_text, encoding="utf-8")

feature_dictionary_text = "# Feature Engineering v0 Feature Dictionary\n\n" + markdown_table(feature_definitions_df) + "\n"
dictionary_report_path.write_text(feature_dictionary_text, encoding="utf-8")

readme_update_text = f"""
# README Update — Feature Engineering v0

Add the following project stage after monitoring threshold calibration.

## Lightweight feature engineering v0

The next project layer creates a transparent feature table from the validated cleaned Google Play review data. This version does not train a sentiment, topic, or machine-learning model.

The feature notebook preserves all {len(features_df):,} cleaned review rows across 10 apps and adds:

- review character and word counts
- low, middle, and high rating groups
- short-review and low-signal flags
- duplicate-identity validation and repeated-text indicators
- review and collection dates
- review age at first collection
- a conservative non-English status with an `undetermined` state
- developer-reply availability
- simple crash/bug, performance/loading, login/account, payment/billing, ads, update/version, and support/service indicators
- app-level review volume, average rating, rating shares, short/low-signal shares, reply share, language shares, and issue shares

The operating sequence remains ingestion, monitoring, and then feature generation. A hard monitoring failure stops downstream use. A warning is reviewed and documented but does not automatically invalidate a completed usable run.

Feature engineering outputs:

```text
notebooks/
└── Google_Play_Review_Feature_Engineering_v0.ipynb

outputs/
├── review_features_v0.csv
├── review_feature_sample_v0.csv
├── app_feature_aggregates_v0.csv
├── feature_definitions_v0.csv
├── feature_quality_assessment_v0.csv
├── feature_validation_checks_v0.csv
├── feature_engineering_source_validation_v0.csv
├── feature_issue_indicator_rates_v0.csv
├── feature_engineering_metadata_v0.json
└── feature_engineering_output_manifest_v0.csv

reports/
├── google_play_feature_engineering_v0_report.md
├── feature_engineering_v0_feature_dictionary.md
└── README_feature_engineering_v0_update.md
```

The most reliable v0 features are identifiers, score/rating group, timestamps, text length, and developer-reply presence. The language, low-signal, repeated-text, and keyword issue fields are documented heuristics and should be used for screening and EDA rather than final labels.
"""
readme_update_path.write_text(textwrap.dedent(readme_update_text).strip() + "\n", encoding="utf-8")

metadata = {
    "project": PROJECT_NAME,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_package": SOURCE_PACKAGE.name,
    "source_package_sha256": sha256_file(SOURCE_PACKAGE),
    "database_file": DB_PATH.name,
    "database_sha256": sha256_file(DB_PATH),
    "review_rows": int(len(features_df)),
    "app_count": int(features_df["app_id"].nunique()),
    "feature_column_count": int(len(features_df.columns)),
    "short_review_char_threshold": SHORT_REVIEW_CHAR_THRESHOLD,
    "low_signal_max_words": LOW_SIGNAL_MAX_WORDS,
    "low_signal_min_alphanumeric_characters": LOW_SIGNAL_MIN_ALNUM_CHARS,
    "rating_group_rule": {"low": "1-2", "middle": "3", "high": "4-5"},
    "duplicate_identity": "source + app_id + review_id",
    "language_method": "conservative deterministic script and marker heuristic",
    "language_undetermined_supported": True,
    "issue_indicator_count": len(issue_flag_columns),
    "model_training_performed": False,
    "monitoring_dependency": "run after ingestion monitoring; hard failure stops downstream use",
    "overall_summary": overall_summary,
}
metadata_output_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Feature outputs written.")
print("Output CSV/JSON files:", len(list(OUTPUT_DIR.iterdir())))
print("Markdown reports:", len(list(REPORT_DIR.iterdir())))


Feature outputs written.
Output CSV/JSON files: 9
Markdown reports: 3


## 14. Verify all deliverables and create the GitHub upload ZIP

The manifest records size and SHA-256 for every required output. The ZIP preserves the repository subfolders so the files can be uploaded directly to the matching GitHub locations.


In [14]:
required_output_paths = [
    review_features_path,
    review_sample_path,
    app_aggregates_path,
    feature_definitions_path,
    feature_quality_path,
    validation_path,
    source_validation_path,
    issue_rates_path,
    metadata_output_path,
    report_path,
    dictionary_report_path,
    readme_update_path,
]

manifest_rows = []
for path in required_output_paths:
    manifest_rows.append({
        "file_name": path.name,
        "relative_path": str(path.relative_to(WORK_DIR)),
        "exists": path.exists(),
        "nonempty": path.exists() and path.stat().st_size > 0,
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "sha256": sha256_file(path) if path.exists() else "",
    })

output_manifest_df = pd.DataFrame(manifest_rows)
if not output_manifest_df[["exists", "nonempty"]].all().all():
    raise ValueError("At least one required feature-engineering output is missing or empty.")

output_manifest_path = OUTPUT_DIR / "feature_engineering_output_manifest_v0.csv"
output_manifest_df.to_csv(output_manifest_path, index=False)

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
(PACKAGE_DIR / "outputs").mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / "reports").mkdir(parents=True, exist_ok=True)

for path in list(OUTPUT_DIR.iterdir()):
    shutil.copy2(path, PACKAGE_DIR / "outputs" / path.name)
for path in list(REPORT_DIR.iterdir()):
    shutil.copy2(path, PACKAGE_DIR / "reports" / path.name)

package_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
package_base = WORK_DIR / f"google_play_feature_engineering_v0_github_upload_files_{package_timestamp}"
package_zip_path = Path(
    shutil.make_archive(
        str(package_base),
        "zip",
        root_dir=PACKAGE_DIR,
    )
)

with zipfile.ZipFile(package_zip_path, "r") as package_zip:
    package_files = sorted(
        name for name in package_zip.namelist() if not name.endswith("/")
    )

expected_package_count = len(list(OUTPUT_DIR.iterdir())) + len(list(REPORT_DIR.iterdir()))
if len(package_files) != expected_package_count:
    raise ValueError("The GitHub upload ZIP file count does not match the generated deliverables.")

print("Feature Engineering v0 completed successfully.")
print("-" * 76)
print("Review-level rows:", f"{len(features_df):,}")
print("App aggregate rows:", len(app_aggregates_df))
print("Sample rows:", len(sample_df))
print("Feature definitions:", len(feature_definitions_df))
print("Validation checks passed:", int(pd.concat([source_guard_df, validation_df, aggregate_validation_df])["passed"].sum()))
print("GitHub upload ZIP:", package_zip_path)
print("ZIP size (MB):", round(package_zip_path.stat().st_size / (1024 ** 2), 2))
display(output_manifest_df[["relative_path", "size_bytes", "sha256"]])

try:
    from google.colab import files
    files.download(str(package_zip_path))
except ImportError:
    pass


Feature Engineering v0 completed successfully.
----------------------------------------------------------------------------
Review-level rows: 34,601
App aggregate rows: 10
Sample rows: 30
Feature definitions: 37
Validation checks passed: 32
GitHub upload ZIP: /mnt/data/feature_engineering_v0_run/google_play_feature_engineering_v0_github_upload_files_20260723_041410_utc.zip
ZIP size (MB): 4.13


,relative_path,size_bytes,sha256
0,outputs/review_features_v0.csv,15447118,7e65a3d282b7121012a940c931bed0a10f32ebf83d8744d1021545dbb87fe2fc
1,outputs/review_feature_sample_v0.csv,7811,2cb5b0cbcbcc285557783c2556f29f39720bdfaa826b22c61ada96d6b05f5d35
2,outputs/app_feature_aggregates_v0.csv,2936,3868bbc549279b42dd4622eebc8eb8a8cc623593b4954df3207d4e61ae3e076d
3,outputs/feature_definitions_v0.csv,8452,e4a04c28fbffc82e978cfdbe3761eb9192a1dd43a79e4e0c547d48bc0ee1e0e8
4,outputs/feature_quality_assessment_v0.csv,1485,194aa43153f1b0b5ac0409e2fd875ff5003f80568efb3b8ef8d46f160a5dfb85
5,outputs/feature_validation_checks_v0.csv,1203,f83f73765cb6ac378c67299c2bc18204b7ab3223529a8862056a2e81125a54f9
6,outputs/feature_engineering_source_validation_v0.csv,4638,bce527d8bf88bbcccd87bd9127998f86848b0a5fdd90039444e0e864a3786dc0
7,outputs/feature_issue_indicator_rates_v0.csv,306,8b5b3a454c1509b3df9257e2a23f651c73722059e596751b4aceaa533f6bd3c2
8,outputs/feature_engineering_metadata_v0.json,1746,2de0c96ca1f78ef134400abd073392a884509e7ce9e56fb328b070bb3ff77131
9,reports/google_play_feature_engineering_v0_report.md,4573,4ce2ac769cc821653acaaaf582cb4545f9958d97a607bc62c27eaba3769ccbbd


## 15. Final conclusion

Feature Engineering v0 preserves the validated review population and adds a small, explainable set of features for later EDA and model planning. No model is trained.

The deterministic and source-provided features are ready for routine descriptive analysis. The language, low-signal, repeated-text, and issue indicators remain explicitly labeled heuristics and should be reviewed before any modeling or automated decision use.
